In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"

silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df = (
    spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
    )

In [0]:
results_selected_df = results_df.drop("url")

In [0]:
results_renamed_df = (
    results_selected_df
    .withColumnsRenamed({
        "raceName": "race_name",
        "constructorId": "constructor_id",
        "driverId": "driver_id",
        "positionText": "position_text",
        "date": "race_date",
        "grid": "grid_position",
        "number": "car_number",
        "position": "final_position",
        "positionText": "final_position_text"
    })  
)

In [0]:
results_distinct_df = (
    results_renamed_df
    .filter(
        F.col("round").isNotNull() &
        F.col("season").isNotNull() &
        F.col("constructor_id").isNotNull() &
        F.col("driver_id").isNotNull()
    )
    .dropDuplicates(["round", "season", "constructor_id", "driver_id"])
)

In [0]:
results_final_df = (
    results_distinct_df
    .withColumn("race_name", F.initcap("race_name"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
)

In [0]:
if not spark.catalog.tableExists(silver_table):
    (
        results_final_df.write
        .format('delta')
        .mode("overwrite")
        .saveAsTable(silver_table)
    )
else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            results_final_df.alias("r"),
            "t.season = r.season AND t.round = r.round AND t.constructor_id = r.constructor_id AND t.driver_id = r.driver_id"
        )
        .whenMatchedUpdate(
            condition="r.batch_id >= t.batch_id",
            set={
                "race_date": "r.race_date",
                "race_name": "r.race_name",
                "grid_position": "r.grid_position",
                "laps": "r.laps",
                "car_number": "r.car_number",
                "points": "r.points",
                "final_position": "r.final_position",
                "final_position_text": "r.final_position_text",
                "status": "r.status",
                "ingestion_timestamp": "r.ingestion_timestamp",
                "source_file": "r.source_file",
                "batch_id": "r.batch_id",
                "updated_at": "r.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )